# Notebook 10: Motion Model Jacobians and Process Noise

This notebook connects EKF prediction math directly to `kiss_slam` motion code and covariance propagation.

> Code connection:
> - `src/kiss_slam/models/motion.py`: `unicycle_jacobians`, `DifferentialDriveMotionModel.jacobians`
> - `src/kiss_slam/ekf_slam.py`: prediction covariance update

## Learning objectives

1. Understand and inspect state Jacobian `F` and control/noise Jacobian `L` (`Fu` in repo).
2. Apply covariance prediction: `P' = F P F^T + L Q L^T`.
3. See how different process noise `Q` changes covariance growth.
4. Tune `Q` to produce target drift behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kiss_slam.models.motion import DifferentialDriveMotionModel
from kiss_slam.types import ControlInput

SEED = 10
np.random.seed(SEED)
print(f"Seed set to {SEED}")

## 1) Jacobians used in EKF prediction

For pose state $x=[x, y, \theta]^T$, control $u=[v,\omega]^T$:

$$
f(x,u)=
\begin{bmatrix}
 x + v\,dt\cos\theta \\n y + v\,dt\sin\theta \\n \theta + \omega dt
\end{bmatrix}
$$

Linearization around current estimate gives:

- `F = df/dx` (state Jacobian, named `fx` in repo),
- `L = df/du` (noise/control Jacobian, named `fu` in repo).

In [ ]:
model = DifferentialDriveMotionModel()
pose = np.array([2.0, -1.0, np.deg2rad(35.0)])
u = ControlInput(v=1.2, w=0.4)
dt = 0.1

F, L = model.jacobians(pose, u, dt)
print("F =")
print(F)
print("\nL =")
print(L)

## 2) Numerical check of Jacobians (sanity)

We can approximate Jacobians with finite differences and compare to analytic values.

In [ ]:
def f_step(x, u, dt):
    return model.predict_state(x, ControlInput(v=float(u[0]), w=float(u[1])), dt)


def finite_diff_jacobians(x, u, dt, eps=1e-6):
    n_x, n_u = x.size, u.size
    F_num = np.zeros((n_x, n_x))
    L_num = np.zeros((n_x, n_u))

    fx0 = f_step(x, u, dt)

    for i in range(n_x):
        dx = np.zeros(n_x)
        dx[i] = eps
        F_num[:, i] = (f_step(x + dx, u, dt) - fx0) / eps

    for i in range(n_u):
        du = np.zeros(n_u)
        du[i] = eps
        L_num[:, i] = (f_step(x, u + du, dt) - fx0) / eps

    return F_num, L_num

x = np.array([0.5, 0.3, -0.7])
u_arr = np.array([1.1, -0.2])
F_ana, L_ana = model.jacobians(x, ControlInput(v=u_arr[0], w=u_arr[1]), dt=0.05)
F_num, L_num = finite_diff_jacobians(x, u_arr, dt=0.05)

print("max |F_ana - F_num|:", np.max(np.abs(F_ana - F_num)))
print("max |L_ana - L_num|:", np.max(np.abs(L_ana - L_num)))

## 3) Covariance propagation in predict step

For robot-only state:
$$
P_{k+1} = F P_k F^T + L Q L^T
$$

- `P`: current state covariance.
- `Q`: control/process noise covariance.

This is exactly the same structure used in the SLAM predict step for the robot block.

In [ ]:
def propagate_covariance(pose, P, control, dt, Q):
    F, L = model.jacobians(pose, control, dt)
    P_next = F @ P @ F.T + L @ Q @ L.T
    return P_next

pose = np.array([0.0, 0.0, 0.0])
P0 = np.diag([1e-4, 1e-4, 1e-4])
control = ControlInput(v=1.0, w=0.4)
dt = 0.1
Q = np.diag([0.05**2, 0.03**2])

P1 = propagate_covariance(pose, P0, control, dt, Q)
print(P1)

## 4) Interactive: covariance growth under different `Q`

We simulate repeated prediction-only steps and compare uncertainty growth.

In [ ]:
def run_predict_only(pose0, P0, control, dt, steps, Q):
    pose = pose0.copy()
    P = P0.copy()
    poses = [pose.copy()]
    traces = [np.trace(P)]

    for _ in range(steps):
        pose = model.predict_state(pose, control, dt)
        P = propagate_covariance(pose, P, control, dt, Q)
        poses.append(pose.copy())
        traces.append(np.trace(P))

    return np.asarray(poses), np.asarray(traces)

pose0 = np.array([0.0, 0.0, 0.0])
P0 = np.diag([1e-4, 1e-4, 1e-4])
control = ControlInput(v=1.0, w=0.5)
dt = 0.1
steps = 120

Q_small = np.diag([0.01**2, 0.005**2])
Q_medium = np.diag([0.05**2, 0.03**2])
Q_large = np.diag([0.12**2, 0.08**2])

_, t_small = run_predict_only(pose0, P0, control, dt, steps, Q_small)
_, t_medium = run_predict_only(pose0, P0, control, dt, steps, Q_medium)
_, t_large = run_predict_only(pose0, P0, control, dt, steps, Q_large)

plt.figure(figsize=(7,4))
plt.plot(t_small, label="small Q")
plt.plot(t_medium, label="medium Q")
plt.plot(t_large, label="large Q")
plt.xlabel("step")
plt.ylabel("trace(P)")
plt.title("Covariance growth vs process noise")
plt.grid(True)
plt.legend();

## 5) Monte Carlo intuition check

Higher `Q` should roughly correspond to larger real trajectory spread when controls are noisy.

In [ ]:
def sample_noisy_rollouts(pose0, control, dt, steps, Q, n_rollouts=200):
    final_positions = []
    for _ in range(n_rollouts):
        pose = pose0.copy()
        for _ in range(steps):
            noise = np.random.multivariate_normal(mean=np.zeros(2), cov=Q)
            u_noisy = ControlInput(v=control.v + noise[0], w=control.w + noise[1])
            pose = model.predict_state(pose, u_noisy, dt)
        final_positions.append(pose[:2])
    return np.asarray(final_positions)

pose0 = np.array([0.0, 0.0, 0.0])
control = ControlInput(v=1.0, w=0.4)
dt = 0.1
steps = 80

pts_small = sample_noisy_rollouts(pose0, control, dt, steps, Q_small)
pts_large = sample_noisy_rollouts(pose0, control, dt, steps, Q_large)

fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(pts_small[:,0], pts_small[:,1], s=10, alpha=0.4, label="small Q")
ax.scatter(pts_large[:,0], pts_large[:,1], s=10, alpha=0.4, label="large Q")
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Final position spread from noisy controls")
ax.grid(True)
ax.legend();

## 6) Exercise: tune `Q` to match target drift

Suppose we want final-position spread around **0.8 m** (roughly) after a fixed run.

### Task
1. Edit `qv_std` and `qw_std`.
2. Simulate rollouts.
3. Compare achieved spread with target.

In [ ]:
# --- student tuning knobs ---
qv_std = 0.07
qw_std = 0.04
Q_try = np.diag([qv_std**2, qw_std**2])

pts = sample_noisy_rollouts(pose0, control, dt, steps, Q_try, n_rollouts=300)
center = pts.mean(axis=0)
radial = np.linalg.norm(pts - center, axis=1)
spread = radial.std()

print(f"Chosen stds: qv={qv_std:.3f}, qw={qw_std:.3f}")
print(f"Estimated spread (std of radial distances): {spread:.3f} m")
print("Target ~0.8 m (example objective)")

plt.figure(figsize=(6,5))
plt.scatter(pts[:,0], pts[:,1], s=10, alpha=0.4)
plt.scatter([center[0]], [center[1]], c="red", label="mean")
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Rollout cloud for chosen Q")
plt.grid(True)
plt.legend();

### Optional solution hint

A bigger `qv_std` mostly stretches distance along the path.
A bigger `qw_std` increases heading uncertainty, often producing wider sideways spread.

Try balancing both rather than only increasing one term.

## Recap

You connected theory and code for EKF prediction:
- derived/inspected `F` and `L` Jacobians,
- verified Jacobians numerically,
- propagated covariance with `P' = FPF^T + LQL^T`,
- observed how `Q` drives covariance growth and drift spread,
- practiced tuning `Q` for a target behavior.